# 🧠 Notebook 05: Proposed Multi-Task Temporal Architecture (MT-TCN-LSTM)
**AI-Enabled Smart Hydroponic Nutrient Balancing System**  
*Sprint 5: Dual-Head Forecasting + Agronomic Stress Classification + MC Dropout Uncertainty*

---

## 1. Executive Summary & Architectural Motivation

Sprint 4 baseline benchmarking yielded three decisive findings:
1. **TCN (B5)** achieved the lowest TDS forecasting error (20.15 ppm) due to dilated causal convolutions providing a large receptive field without recurrent gradient bottlenecks.
2. **Vanilla LSTM (B2)** achieved the highest overall $R^2$ (0.532) and best pH stability through global recurrent hidden state persistence.
3. **Single-task continuous regressors (B1–B7)** suffer from lack of domain-aware regularization and **completely lack an explicit stress classification mechanism**.

To resolve these limitations, we propose the **Multi-Task Dilated Temporal Convolutional LSTM Network (MT-TCN-LSTM)**:

```
Input Tensor: (N, 15, 5) — 150 seconds lookback across 5 sensors
         ↓
┌────────────────────────────────────────────────────────┐
│      Multi-Scale Dilated Causal Conv1D Residual Blocks  │ (d=1, 2, filters=64)
└──────────────────────────┬─────────────────────────────┘
                           ↓
┌────────────────────────────────────────────────────────┐
│           Recurrent LSTM Sequence Aggregator            │ (units=64, Dropout=0.3)
└──────────────────────────┬─────────────────────────────┘
                           ↓
                 Shared Representation
                ┌──────────┴──────────┐
                ▼                     ▼
      ┌──────────────────┐  ┌──────────────────┐
      │  Head 1: Dense   │  │  Head 2: Dense   │
      │   Forecasting    │  │ Stress Detection │
      │ (5 sensor states)│  │ (Binary Sigmoid) │
      └──────────────────┘  └──────────────────┘
```

### Multi-Task Loss Formulation
$$\mathcal{L}_{\text{total}} = (1 - \lambda) \cdot \text{MSE}_{\text{forecast}} + \lambda \cdot \text{BCE}_{\text{stress}}, \quad \lambda = 0.2$$

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Ensure project root is in path
ROOT_DIR = os.path.dirname(os.path.abspath(''))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.config import RAW_CSV, FEATURES, TARGET_COLS, TIME_STEPS, TOLERANCES, SAFE_RANGES
from src.models import build_proposed_model
from src.uncertainty import predict_with_mc_dropout, evaluate_uncertainty_calibration

print("All modules loaded successfully.")

## 2. Load Proposed Model Experimental Records
We load the structured experimental results generated by `notebooks/05_proposed_model.py`.

In [ ]:
exp_file = os.path.join(ROOT_DIR, 'experiments', 'exp_005_proposed_model.json')
if os.path.exists(exp_file):
    with open(exp_file, 'r', encoding='utf-8') as f:
        exp_data = json.load(f)
    
    reg = exp_data['regression_metrics']
    cls_m = exp_data['classification_metrics']
    print(f"Experiment ID: {exp_data['experiment_id']}")
    print(f"Architecture:  {exp_data['architecture']['name']} ({exp_data['architecture']['params']:,} parameters)")
    print(f"Inference Latency: {exp_data['architecture']['latency_ms']} ms")
    print(f"Average Forecasting R²: {reg['aggregate']['mean_R2']:.4f}")
    print(f"Stress ROC-AUC Score:   {exp_data['stress_roc_auc']:.4f}")
else:
    print(f"File {exp_file} not found. Please execute notebooks/05_proposed_model.py first.")

## 3. Forecasting Performance Comparison (Proposed vs Baselines)
Comparing the proposed model against B0, B1, B2, and B5 on the standardized chronological test set.

In [ ]:
bench_file = os.path.join(ROOT_DIR, 'reports', 'benchmark_table.md')
if os.path.exists(bench_file):
    with open(bench_file, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Benchmark table not found.")

## 4. Visualizations: Performance, Stress Classification, & Uncertainty Bands
Displaying publication-quality figures generated in Sprint 5.

In [ ]:
fig_dir = os.path.join(ROOT_DIR, 'reports', 'figures')
fig_list = [
    ("Figure 16: Proposed Model vs Top Baselines (MAE & R²)", "fig16_proposed_vs_baselines_mae_r2.png"),
    ("Figure 17: Stress Detection Head Confusion Matrix & Probability Distribution", "fig17_stress_confusion_matrix.png"),
    ("Figure 18: Monte Carlo Dropout 90% Confidence Bands & Epistemic Uncertainty", "fig18_mc_dropout_uncertainty_bands.png"),
]

for title, fname in fig_list:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"### {title}")
        display(Image(filename=fpath))
    else:
        print(f"Figure {fname} not found yet.")

## 5. Monte Carlo Dropout Uncertainty & Safety Layer Integration

The model outputs an epistemic standard deviation $\sigma_{\text{MC}}$ alongside each forecast:
- When $\sigma < \sigma_{\text{medium}}$, autonomous dosing actuators are authorized.
- When $\sigma > \sigma_{\text{high}}$, the system locks out chemical dosing and issues an audible operator alert.

This provides the missing safety guarantee required for real-world automated hydroponic management.